In [15]:
# ============================================================
# SQL ANALYSIS OF E-COMMERCE DATASET
# ============================================================

!pip install -q pandas kagglehub

import pandas as pd
import sqlite3
import kagglehub
import os

# ------------------------------------------------------------
# 1. LOAD AND CLEAN DATASET
# ------------------------------------------------------------
print("Loading dataset...")
dataset_url = "prince7489/e-commerce-sales"

try:
    path = kagglehub.dataset_download(dataset_url)
    for f in os.listdir(path):
        if f.endswith('.csv'):
            csv_path = os.path.join(path, f)
            break
    df = pd.read_csv(csv_path)
    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"Kaggle failed: {e}\nCreating synthetic dataset...")
    import numpy as np
    np.random.seed(42)
    df = pd.DataFrame({
        'Order ID': range(1, 1001),
        'Order Date': pd.date_range('2024-01-01', periods=1000, freq='D'),
        'Customer Name': [f'Cust_{i}' for i in range(1, 1001)],
        'Region': np.random.choice(['North','South','East','West'], 1000),
        'City': np.random.choice(['Delhi','Mumbai','Bangalore','Kolkata'], 1000),
        'Category': np.random.choice(['Electronics','Clothing','Home','Books','Toys'], 1000),
        'Sub-Category': np.random.choice(['Mobile','Laptop','Shirt','Sofa','Novel'], 1000),
        'Product Name': [f'Prod_{i}' for i in range(1, 1001)],
        'Quantity': np.random.randint(1, 10, 1000),
        'Unit Price': np.random.uniform(100, 5000, 1000).round(2),
        'Discount': np.random.uniform(0, 0.3, 1000).round(2),
        'Sales': np.random.uniform(100, 20000, 1000).round(2),
        'Profit': np.random.uniform(-1000, 5000, 1000).round(2),
        'Payment Mode': np.random.choice(['Credit Card','Debit Card','UPI'], 1000)
    })

# Remove spaces from column names for SQL compatibility
df.columns = df.columns.str.replace(' ', '_')
print("Columns:", list(df.columns))

# ------------------------------------------------------------
# 2. CREATE SQLITE DATABASE
# ------------------------------------------------------------
conn = sqlite3.connect(':memory:')
table_name = "sales_data"
df.to_sql(table_name, conn, if_exists='replace', index=False)
print(f"Table '{table_name}' created with {len(df)} rows.\n")

# ------------------------------------------------------------
# 3. HELPER FUNCTION: RUN QUERY AND PRINT TABLE
# ------------------------------------------------------------
def run_query(sql, title):
    print(f"\n{title}")
    print("-" * 50)
    try:
        result = pd.read_sql_query(sql, conn)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"Error: {e}")

# ------------------------------------------------------------
# 4. IMPORTANT SQL QUERIES (12 queries)
# ------------------------------------------------------------

queries = [
    # 1. Total orders
    ("1. Total number of orders",
     f"SELECT COUNT(*) AS total_orders FROM {table_name}"),

    # 2. Sales by category (top 5)
    ("2. Top 5 categories by total sales",
     f"SELECT Category, SUM(Sales) AS total_sales FROM {table_name} GROUP BY Category ORDER BY total_sales DESC LIMIT 5"),

    # 3. Average profit per order
    ("3. Average profit per order",
     f"SELECT AVG(Profit) AS avg_profit FROM {table_name}"),

    # 4. Unique regions
    ("4. All regions",
     f"SELECT DISTINCT Region FROM {table_name} ORDER BY Region"),

    # 5. Region with highest total sales
    ("5. Region with highest sales",
     f"SELECT Region, SUM(Sales) AS total_sales FROM {table_name} GROUP BY Region ORDER BY total_sales DESC LIMIT 1"),

    # 6. Monthly sales trend (first 6 months)
    ("6. Monthly sales trend (first 6 months)",
     f"SELECT strftime('%Y-%m', Order_Date) AS month, SUM(Sales) AS monthly_sales FROM {table_name} GROUP BY month ORDER BY month LIMIT 6"),

    # 7. Profit by region
    ("7. Total profit by region",
     f"SELECT Region, SUM(Profit) AS total_profit FROM {table_name} GROUP BY Region ORDER BY total_profit DESC"),

    # 8. Discount impact on average sales
    ("8. Discount impact",
     f"SELECT CASE WHEN Discount > 0.2 THEN 'High Discount (>20%)' ELSE 'Low Discount (≤20%)' END AS discount_level, AVG(Sales) AS avg_sales FROM {table_name} GROUP BY discount_level"),

    # 9. Top 5 products by sales
    ("9. Top 5 products by sales",
     f"SELECT Product_Name, SUM(Sales) AS total_sales FROM {table_name} GROUP BY Product_Name ORDER BY total_sales DESC LIMIT 5"),

    # 10. Payment mode preference
    ("10. Order count by payment mode",
     f"SELECT Payment_Mode, COUNT(*) AS order_count FROM {table_name} GROUP BY Payment_Mode"),

    # 11. Average order value by customer segment (using Region as proxy)
    ("11. Average sales by region",
     f"SELECT Region, AVG(Sales) AS avg_order_value FROM {table_name} GROUP BY Region ORDER BY avg_order_value DESC"),

    # 12. Products with negative profit
    ("12. Products with negative profit (loss)",
     f"SELECT Product_Name, Profit FROM {table_name} WHERE Profit < 0 ORDER BY Profit LIMIT 10")
]

# ------------------------------------------------------------
# 5. EXECUTE ALL QUERIES AND DISPLAY TABLES
# ------------------------------------------------------------
for title, sql in queries:
    run_query(sql, title)

# ------------------------------------------------------------
# 6. CLOSE CONNECTION
# ------------------------------------------------------------
conn.close()
print("\nAll queries executed successfully.")

Loading dataset...
Using Colab cache for faster access to the 'e-commerce-sales' dataset.
Dataset loaded: 5000 rows, 14 columns
Columns: ['Order_ID', 'Order_Date', 'Customer_Name', 'Region', 'City', 'Category', 'Sub-Category', 'Product_Name', 'Quantity', 'Unit_Price', 'Discount', 'Sales', 'Profit', 'Payment_Mode']
Table 'sales_data' created with 5000 rows.


1. Total number of orders
--------------------------------------------------
 total_orders
         5000

2. Top 5 categories by total sales
--------------------------------------------------
  Category  total_sales
Home Decor  57233222.35
 Furniture  56647187.90
  Clothing  55053908.30
     Books  54932643.00
   Kitchen  54227902.30

3. Average profit per order
--------------------------------------------------
  avg_profit
15941.746982

4. All regions
--------------------------------------------------
Region
  East
 North
 South
  West

5. Region with highest sales
--------------------------------------------------
Region  total_